##  TechMind — Exploración y Preparación del Dataset

## Equipo tejONEs

## SEMANA 1
Este pipeline corresponde a la realización de las actividades para la semana 1:
- [x]  Carga y normalización de los datasets:
  - [Kaggle - Coursera Courses dataset 2021](https://www.kaggle.com/datasets/khusheekapoor/coursera-courses-dataset-2021)
  - [Microsoft Learn dataset - Extracción hecha por el equipo](https://drive.google.com/drive/folders/17vrew3ifV7DacKXM0ytEsanfvFrPGVTV?usp=drive_link)

- [x]  Limpieza de texto (HTML, URLs, duplicados, filtro de mínimo 30 palabras)
- [x]  Mapeo de categorías por palabras clave (Backend, Frontend, Data Science, DevOps, Bases de Datos, Mobile, Cloud)
- [x]  Balanceo de datos (máximo 50 registros por categoría)
- [x]  Traducción a español (título, texto) con manejo de errores visible
- [x]  Exportación del dataset final, sin columnas duplicadas, en carpeta `procesados/`

# 0.Configuración e importaciones

In [37]:
import os
import re
from pathlib import Path

import nltk
import pandas as pd
from deep_translator import GoogleTranslator
from nltk.corpus import stopwords
from tqdm.notebook import tqdm

# Descargar recursos de NLP
nltk.download('stopwords', quiet=True)
spanish_stopwords = set(stopwords.words('spanish'))


def find_project_root(start: Path) -> Path:
    """Encuentra la raíz del proyecto desde el directorio actual o ubicaciones comunes de Windows."""
    for candidate in [start, *start.parents]:
        if (candidate / 'data_science').exists() and (candidate / 'README.md').exists():
            return candidate

    common_roots = [
        Path.home() / 'Documents',
        Path.home() / 'OneDrive' / 'Documents',
        Path.home() / 'Desktop',
    ]

    for base in common_roots:
        if not base.exists():
            continue
        repo_dir = base / 'G9-LATAM-Team-25'
        if (repo_dir / 'data_science').exists() and (repo_dir / 'README.md').exists():
            return repo_dir

    return start


def resolve_path(env_key, fallback):
    value = os.getenv(env_key)
    if value:
        candidate = Path(value).expanduser()
        if not candidate.is_absolute():
            candidate = (project_root / candidate).resolve()
        if candidate.exists():
            return candidate.resolve()
    return fallback.resolve() if isinstance(fallback, Path) else Path(fallback).resolve()


base_dir = Path.cwd().resolve()
project_root = find_project_root(base_dir)

data_dir = resolve_path('DATA_DIR', project_root / 'data_science' / 'data')
raw_folder = resolve_path('RAW_FOLDER', data_dir / 'crudos')
processed_folder = resolve_path('PROCESSED_FOLDER', data_dir / 'procesados')

PROJECT_FOLDER = data_dir
RAW_FOLDER = raw_folder
PROCESSED_FOLDER = processed_folder



RAW_FOLDER.mkdir(parents=True, exist_ok=True)
print(f'✅ Ruta de procesados existe: {Path(PROCESSED_FOLDER).exists()}')
PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)
print(f'✅ Ruta de crudos existe: {Path(RAW_FOLDER).exists()}')
print(f'✅ Ruta de datos existe: {Path(PROJECT_FOLDER).exists()}')
PROJECT_FOLDER = str(PROJECT_FOLDER.resolve())
print(f'📂 Datos procesados: {PROCESSED_FOLDER}')
RAW_FOLDER = str(RAW_FOLDER.resolve())
print(f'📂 Datos crudos: {RAW_FOLDER}')
PROCESSED_FOLDER = str(PROCESSED_FOLDER.resolve())
print(f'📁 Proyecto local: {PROJECT_FOLDER}')


✅ Ruta de procesados existe: True
✅ Ruta de crudos existe: True
✅ Ruta de datos existe: True
📂 Datos procesados: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\procesados
📂 Datos crudos: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\crudos
📁 Proyecto local: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data


# 1.Coursera Courses dataset 2021

##  1.1 Carga de datos y normalización

In [39]:
raw_candidates = [
    Path(RAW_FOLDER) / 'dataset_coursera.csv',
    Path(PROJECT_FOLDER) / 'crudos' / 'dataset_coursera.csv',
    Path(project_root) / 'data_science' / 'data' / 'crudos' / 'dataset_coursera.csv',
]
raw_file = next((candidate for candidate in raw_candidates if candidate.exists()), None)
file_path = str(raw_file) if raw_file else str(raw_candidates[0])

if raw_file is None:
    raise FileNotFoundError(
        f"No se encontró el archivo: {raw_candidates[0]}\n"
        f"Verifica que exista en cualquiera de estas rutas:\n"
        + "\n".join(str(path) for path in raw_candidates)
    )

# Carga segura con alternativa de codificación (encoding fallback) en caso de error
try:
    df = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='latin1')

# Renombrar columnas para cumplir estrictamente con el esquema de la base de datos
df = df.rename(columns={
    'Course Name': 'titulo',
    'Course Description': 'texto',
    'Skills': 'categoria_original',
    'University': 'autor',
})[['titulo', 'texto', 'categoria_original', 'autor']].copy()

# 'tipo' respeta el esquema del equipo: "texto" (tecleado a mano, vía POST /contenido)

# o "articulo" (documento con fuente, vía carga de archivo). Este dataset son

# descripciones de cursos - documentos, no texto tecleado a mano - así que correspondedisplay(df.sample(3))

# 'articulo'.print("\n--- Muestra aleatoria de los datos cargados ---")

df['tipo'] = 'articulo'

print(f"✅ Datos cargados correctamente. Filas totales: {len(df)}")


✅ Datos cargados correctamente. Filas totales: 3522


## 1.2 Limpieza inicial del texto

In [40]:
def clean_html_urls(text):
    text = re.sub(r'<[^>]+>', ' ', str(text))
    return re.sub(r'http\S+', '', text)

df['texto'] = df['texto'].apply(clean_html_urls)
df['titulo'] = df['titulo'].apply(clean_html_urls)

# Eliminar duplicados basados en la descripción del curso
initial_count = len(df)
df = df.drop_duplicates(subset='texto').reset_index(drop=True)
print(f"Duplicados eliminados: {initial_count - len(df)} (Filas restantes: {len(df)})")

# Filtro de calidad: mínimo 30 palabras (no caracteres). Un umbral de caracteres
# es demasiado permisivo — 20 caracteres son apenas 3-4 palabras, casi no filtra nada.
initial_count = len(df)
df = df[df['texto'].str.split().str.len() >= 30].reset_index(drop=True)
print(f"Textos con menos de 30 palabras eliminados: {initial_count - len(df)} (Filas restantes: {len(df)})")

print("✅ Limpieza de HTML, URLs, duplicados y textos cortos completada.")

Duplicados eliminados: 125 (Filas restantes: 3397)
Textos con menos de 30 palabras eliminados: 43 (Filas restantes: 3354)
✅ Limpieza de HTML, URLs, duplicados y textos cortos completada.


## 1.3. Mapeo de categorías (Regex por palabras clave)


In [41]:
keywords_mapping = {
    'Backend': ['backend','java', 'spring', 'c#', 'php', 'node.js', 'django', 'perl', 'ruby', 'scala', 'clojure', 'rust', 'haskell', 'elixir', 'earlang', 'flask', 'typescript', 'node', 'nodejs', 'laravel', 'api', 'rest', 'backend', 'csharp', 'dotnet', 'blockchain', 'cplusplus', 'graphql', 'kafka', 'solr', 'rabbitMQ', 'nginx', 'openresty', 'nestjs', 'firebase', '.net', 'rails'],
    'Bases de Datos': ['data base','data bases','bd','sql', 'mysql', 'mongodb', 'postgresql', 'redis', 'oracledb', 'cassandra', 'couchdb', 'hive', 'realm', 'mariadb', 'cockroachdb', 'elasticsearch', 'sqlite', 'mssql', 'sql server', 'sqlserver', 'cosmos db', 'database', 'bases de datos', 'nosql'],
    'Cloud': ['cloud','aws', 'azure', 'oraclecloud', 'oci', 'googlecloud', 'gcp', 'nube', 'cloud', 'virtual machine'],
    'Data Science': ['data science','python', 'pandas', 'machine learning', 'excel','bigquery', 'llm', 'r', 'ia', 'deep learning', 'tensorflow', 'pytorch', 'numpy', 'seaborn', 'matplotlib', 'opencv', 'scikitlearn', 'scikit learn','scikit-learn', 'd3js', 'chartjs', 'canvasjs', 'kibana', 'grafana', 'artificial intelligence', 'data mining','data analytics', 'data analysis','data modeling','powerbi','tableau'],
    'DevOps': ['devops','docker', 'kubernetes', 'ci/cd', 'devops', 'git', 'github', 'jenkins', 'bash', 'travisci', 'circleci', 'containers'],
    'Frontend': ['frontend','javascript', 'html', 'html5', 'css', 'css3', 'react', 'angular', 'angularjs', 'vue', 'vuejs', 'scratch', 'frontend', 'svelte', 'backbonejs', 'bootstrap', 'vuetify', 'pug', 'gulp', 'sass', 'redux', 'webpack', 'babel', 'tailwind', 'materialize', 'bulma', 'gtk', 'qt', 'quasar', 'wxwidgets', 'wx widgets', 'ember', 'blazor wasm','blazor webassembly'],
    'Mobile': ['objectivec', 'objective-c', 'android', 'ios', 'flutter', 'kotlin', 'swift', 'dart', 'nativescript', 'xamarin', 'reactnative', 'react native', 'ionic', 'apachecordova', 'mobile', 'multiplatform']
}

# Aplanamos el diccionario para facilitar la búsqueda
category_mapping = {}
for category, keywords in keywords_mapping.items():
    for kw in keywords:
        category_mapping[kw.lower()] = category

def _find_category_in_text(text):
    text = str(text).lower()
    # Ordenar de mayor a menor longitud para evitar que claves cortas coincidan dentro de largas
    sorted_keys = sorted(category_mapping.items(), key=lambda x: -len(x[0]))
    for key, category in sorted_keys:
        # \b asegura la coincidencia exacta de la palabra
        pattern = r'\b' + re.escape(key) + r'\b'
        if re.search(pattern, text):
            return category, key
    return None, None

def assign_category_from_multiple_cols(row):
    # 1. Intentar mapear desde 'titulo' y 'texto' combinados
    combined_text = (str(row['titulo']) + ' ' + str(row['texto'])).lower()
    category, keyword = _find_category_in_text(combined_text)
    if category is not None:
        return category, keyword

    # 2. Si no se encuentra, intentar mapear desde 'categoria_original'
    category_orig = str(row['categoria_original']).lower()
    category, keyword = _find_category_in_text(category_orig)
    if category is not None:
        return category, keyword

    return None, None

# Aplicamos la función usando la concatenación de título y texto, y luego categoria_original para mayor cobertura
results = df.apply(assign_category_from_multiple_cols, axis=1)
df[['categoria', 'palabra_clave']] = pd.DataFrame(results.tolist(), index=df.index)

# Diagnóstico: tasa de coincidencia
total_rows = len(df)
matched_rows = df['categoria'].notna().sum()
print(f" Categorías mapeadas con éxito: {matched_rows} de {total_rows} ({matched_rows/total_rows*100:.1f}%)" if total_rows > 0 else "No hay datos para procesar.")

if total_rows > 0 and (matched_rows / total_rows) < 0.3:
    print("⚠️ ALERTA: La tasa de coincidencia es menor al 30%. Revisa el diccionario de mapeo.")

df = df.dropna(subset=['categoria']).reset_index(drop=True)

print("\n✅ Categorías asignadas.")
print("--- Distribución resultante tras el mapeo ---")
print(df['categoria'].value_counts())

print("\n--- Muestra de validación (Keyword vs Categoría) ---")
display(df[['titulo','texto', 'palabra_clave', 'categoria','categoria_original']].sample(min(10, len(df))))

 Categorías mapeadas con éxito: 1392 de 3354 (41.5%)

✅ Categorías asignadas.
--- Distribución resultante tras el mapeo ---
categoria
Data Science      783
Backend           161
Mobile            110
Frontend          107
Cloud              94
Bases de Datos     83
DevOps             54
Name: count, dtype: int64

--- Muestra de validación (Keyword vs Categoría) ---


,titulo,texto,palabra_clave,categoria,categoria_original
779,Cities are back in town : sociologie urbaine p...,La mondialisation et l'europ�anisation favoris...,r,Data Science,india City Planning law constitution justi...
310,Introduction to Natural Language Processing in...,"In this 1-hour long project-based course, you ...",python,Data Science,Language Learning Python Programming Experim...
1196,Functional Programming Principles in Scala,Functional programming is becoming increasingl...,javascript,Frontend,functional programming ?-recursive function ...
581,Cloud Security Basics,This course introduces you to cybersecurity fo...,cloud,Cloud,separation of duties protocol stack cloud co...
904,Programming for the Internet of Things Project,"In this Capstone course, you will design a mic...",python,Data Science,internet Computer Programming internet of th...
411,A Scientific Approach to Innovation Management,How can innovators understand if their idea is...,data analysis,Data Science,Regression Regression Analysis analysis inn...
157,Creating Models using Smartpls,"In this 1-hour long project-based course, you ...",data analysis,Data Science,modeling project mine show me! running Dat...
473,Handheld AR App Development with Unity,"Augmented Reality, or AR, will transform how w...",android,Mobile,physics augmented reality iOS Development a...
312,Predicting Salaries with Simple Linear Regress...,"In this 1-hour long project-based course, you ...",r,Data Science,line fitting R Programming project Machine ...
1235,Create Python Linux Script to Generate a Disk ...,There are many choices when it comes to writin...,python,Data Science,directory structure command (computing) web ...


##  1.4. Balanceo de datos (Máximo 50 registros por categoría)

In [42]:
df = pd.concat([
    group.sample(min(len(group), 50), random_state=42)
    for _, group in df.groupby('categoria')
]).reset_index(drop=True)

print("✅ Datos balanceados (Máximo 50 registros por categoría).")

✅ Datos balanceados (Máximo 50 registros por categoría).


##  1.5. Traducción y procesamiento NLP

In [43]:
tqdm.pandas()

translation_errors = []

def translate_to_spanish(text):
    try:
        return GoogleTranslator(source='en', target='es').translate(text[:1500])
    except Exception as e:
        translation_errors.append(type(e).__name__)
        return ""

# Traduce título y texto principal al español
df['titulo_es'] = df['titulo'].progress_apply(translate_to_spanish)
df['texto_es'] = df['texto'].progress_apply(translate_to_spanish)

if translation_errors:
    from collections import Counter
    print(f"⚠️ {len(translation_errors)} traducciones fallaron. Tipos de error: {Counter(translation_errors)}")

def clean_nlp(text):
    # Quita signos de puntuación y pasa a minúsculas
    text = re.sub(r'[^\w\sáéíóúñ]', ' ', text.lower())
    # Elimina stopwords en español y palabras muy cortas
    return ' '.join([word for word in text.split() if word not in spanish_stopwords and len(word) > 2])

df['texto_limpio'] = df['texto_es'].apply(clean_nlp)

# Elimina filas donde la traducción del texto falló
initial_count = len(df)
df = df[df['texto_es'].str.strip() != ''].reset_index(drop=True)
print(f"\nTraducciones fallidas eliminadas: {initial_count - len(df)} (Filas restantes: {len(df)})")

# Guarda un respaldo
df.to_csv(f'{PROCESSED_FOLDER}/translation_backup.csv', index=False)
print("✅ Traducción completada y respaldo guardado.")

# Reemplaza las columnas originales y descarta las auxiliares
df['titulo'] = df['titulo_es']
df['texto'] = df['texto_es']
# Solo nos quedamos con las columnas del esquema solicitado
df = df[['titulo', 'texto', 'categoria', 'autor', 'tipo']]

print("✅ Columnas consolidadas")

  0%|          | 0/350 [00:00<?, ?it/s]

  0%|          | 0/350 [00:00<?, ?it/s]


Traducciones fallidas eliminadas: 0 (Filas restantes: 350)
✅ Traducción completada y respaldo guardado.
✅ Columnas consolidadas


##  1.6.Exportación final y auditoría de calidad

In [44]:
final_df = df.copy()

# Estructura final: titulo, texto, categoria, autor, tipo
final_df = final_df[['titulo', 'texto', 'categoria', 'autor', 'tipo']]

print("=== DISTRIBUCIÓN FINAL POR CATEGORÍA ===")
print("   === DATASET - COURSERA 2021 ===")
category_counts = final_df['categoria'].value_counts()
print(category_counts)

# Alerta de umbral mínimo
print("\n--- Validación de Umbral Mínimo ---")
for cat, count in category_counts.items():
    if count < 30:
        print(f"⚠️ ADVERTENCIA: '{cat}' tiene {count} registros (Mínimo requerido: 30).")

final_df.to_csv(f'{PROCESSED_FOLDER}/dataset_FINAL_coursera.csv', index=False)
print("\n✅ Dataset Coursera exportado con el nuevo esquema.")



display(final_df.head())

=== DISTRIBUCIÓN FINAL POR CATEGORÍA ===
   === DATASET - COURSERA 2021 ===
categoria
Backend           50
Bases de Datos    50
Cloud             50
Data Science      50
DevOps            50
Frontend          50
Mobile            50
Name: count, dtype: int64

--- Validación de Umbral Mínimo ---

✅ Dataset Coursera exportado con el nuevo esquema.


,titulo,texto,categoria,autor,tipo
0,"Implementación, depuración y rendimiento de ap...","En este curso, los desarrolladores de aplicaci...",Backend,Google Cloud,articulo
1,Análisis de oportunidades de blockchain,En este cuarto y último curso de especializaci...,Backend,INSEAD,articulo
2,Explotación y protección de vulnerabilidades e...,"En este curso, desempeñaremos muchos papeles. ...",Backend,"University of California, Davis",articulo
3,El árbol Merkle y las criptomonedas,Aplique lo que ha aprendido sobre criptografía...,Backend,"University of California, Irvine",articulo
4,Transformación Digital,"La transformación digital es un tema candente,...",Backend,University of Virginia,articulo


# 2.Microsoft Learn API dataset

##  2.1 Carga de datos y normalización

In [45]:
file_path = f'{RAW_FOLDER}/courses_mslearn.csv'

# Carga segura con alternativa de codificación (encoding fallback) en caso de error
try:
    df_mslearn = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df_mslearn = pd.read_csv(file_path, encoding='latin1')

print(f"✅ Datos cargados correctamente. Filas totales: {len(df_mslearn)}")
print("\n--- Muestra aleatoria de los datos cargados ---")
display(df_mslearn.sample(3))

✅ Datos cargados correctamente. Filas totales: 4606

--- Muestra aleatoria de los datos cargados ---


,id,titulo,descripcion,categoria,subcategoria,temario,tecnologias_mencionadas,tipo_contenido,nivel_dificultad,idioma,fuente,autor,url,fecha_publicacion,licencia,fecha_recoleccion
1603,1604,Planeamiento e implementación de la seguridad ...,Planeamiento e implementación de la seguridad ...,Seguridad,NaN,Introducción | Planee e implemente la segurida...,"api, app service, application gateway, azure, ...",módulo,Intermedio,es,microsoft learn,Microsoft,https://learn.microsoft.com/es-es/training/mod...,2026-07-14,Información protegida (derechos de autor),2026-07-20
3325,3326,Supervisión y solución de problemas de sistema...,La solución de problemas es una tarea importan...,Aplicaciones empresariales,Comunicación,Introducción | Diagnóstico y solución de probl...,"microsoft 365, microsoft teams, teams",módulo,Avanzado,es,microsoft learn,Microsoft,https://learn.microsoft.com/es-es/training/mod...,2026-04-13,Información protegida (derechos de autor),2026-07-20
1403,1404,Creación de flujos de trabajo de integración c...,Aprenda a crear flujos de trabajo para agregar...,Desarrollo de aplicaciones,DevOps,Introducción | ¿Cómo puedo usar las Acciones d...,"azure, github",módulo,Principiante,es,microsoft learn,Microsoft,https://learn.microsoft.com/es-es/training/mod...,2026-02-19,Información protegida (derechos de autor),2026-07-20


In [46]:
df_mslearn.columns

Index(['id', 'titulo', 'descripcion', 'categoria', 'subcategoria', 'temario',
       'tecnologias_mencionadas', 'tipo_contenido', 'nivel_dificultad',
       'idioma', 'fuente', 'autor', 'url', 'fecha_publicacion', 'licencia',
       'fecha_recoleccion'],
      dtype='str')

Para ajustarse a la estructura del dataset con las columnas:
`titulo, texto, categoría, autor, tipo`

In [47]:
# Crear la columna 'texto' con descripcion concatenada a la columna 'temario'(limpiando los separadores '|')
df_mslearn['texto'] = df_mslearn['descripcion'] + ' ' + df_mslearn['temario'].str.replace('|', ' ', regex=False)


# Renombrar columnas para ajustarse al esquema
df_mslearn = df_mslearn.rename(columns={'tipo_contenido': 'tipo'})

# Seleccionar las columnas requeridas y mantener las auxiliares solicitadas
columnas_finales = ['titulo', 'texto','categoria', 'subcategoria', 'autor', 'tipo', 'tecnologias_mencionadas']
df_mslearn = df_mslearn[columnas_finales]

print("✅ Estructura ajustada correctamente.")
print(f"Columnas actuales: {df_mslearn.columns.tolist()}")
display(df_mslearn.head(3))

✅ Estructura ajustada correctamente.
Columnas actuales: ['titulo', 'texto', 'categoria', 'subcategoria', 'autor', 'tipo', 'tecnologias_mencionadas']


,titulo,texto,categoria,subcategoria,autor,tipo,tecnologias_mencionadas
0,Experimento con Azure Machine Learning,Obtenga información sobre cómo encontrar el me...,Inteligencia artificial,Aprendizaje automático,Microsoft,módulo,"azure, datos, ia, learning, machine, machine l..."
1,Implementación de la protección de datos del d...,En este módulo se describe cómo puede usar Int...,NaN,NaN,Microsoft,módulo,"configuration-manager, datos, intune, windows"
2,Adición de lógica de decisión al código median...,Aprenda a bifurcar la ruta de acceso de ejecuc...,Desarrollo de aplicaciones,NaN,Microsoft,módulo,".net, c#, visual studio code"


Pero por el momento se mantienen `subcategoria` y `tecnologias_mencionadas` para mejorar la categorización del contenido.

##  1.2 Limpieza inicial del texto

Se detectaron que los ultimos 8 registros tienen título en inglés. Se borraran para mantener el contenido en español.

In [48]:
df_mslearn.tail(8)

,titulo,texto,categoria,subcategoria,autor,tipo,tecnologias_mencionadas
4598,Architect production-grade multi-agent AI solu...,Learn how to design production-grade agentic A...,Inteligencia artificial,NaN,Microsoft,ruta de aprendizaje,"agent-framework, azure, azure-managed-redis, c..."
4599,"Monitor, evaluate, and operate multi-agent AI ...",Learn how to operate production multi-agent so...,Inteligencia artificial,NaN,Microsoft,ruta de aprendizaje,"agent-framework, azure, azure-managed-redis, c..."
4600,Minecraft Esports Teacher Academy,"Learn the role of esports in education, how to...",NaN,NaN,Microsoft,ruta de aprendizaje,minecraft
4601,Introduction to student security operations ce...,Learn what security operations centers (SOCs) ...,NaN,NaN,Microsoft,ruta de aprendizaje,"copilot, microsoft, microsoft sentinel, micros..."
4602,Introduction to AI literacy,Using the Empowering Learners for the Age of A...,NaN,NaN,Microsoft,ruta de aprendizaje,"ia, m365-education, microsoft, ms-copilot"
4603,Learning Accelerators for educators,Bring individualized learning to the classroom...,NaN,NaN,Microsoft,ruta de aprendizaje,"datos, desarrollo, learning, m365-education, m..."
4604,Troubleshoot Active Directory Domain Services ...,Diagnose Active Directory Domain Services repl...,Seguridad,Identidad y acceso,Microsoft,módulo,"data, windows server"
4605,From prompts to goals,Learn how to apply Copilot Cowork to everyday ...,Inteligencia artificial,NaN,Microsoft,módulo,"copilot, m365-apps, microsoft 365, ms-copilot"


In [49]:
df_mslearn = df_mslearn.iloc[:-8].reset_index(drop=True)

print(f"✅ Registros eliminados. Filas actuales: {len(df_mslearn)}")
display(df_mslearn.tail(3))

✅ Registros eliminados. Filas actuales: 4598


,titulo,texto,categoria,subcategoria,autor,tipo,tecnologias_mencionadas
4595,Diseño del Aprendizaje en el Siglo XXI,El Diseño del Aprendizaje en el Siglo XXI para...,NaN,NaN,Microsoft,ruta de aprendizaje,office 365
4596,Marco de Transformación Educativa,"Las formas en que las personas interactúan, so...",NaN,NaN,Microsoft,ruta de aprendizaje,"microsoft, office 365"
4597,Aprendizaje en contexto remoto con uso de Teams,El aprendizaje en contexto remoto se materiali...,NaN,NaN,Microsoft,ruta de aprendizaje,"microsoft teams, teams"


Se aplica la limpieza

In [50]:
def clean_html_urls(text):
    text = re.sub(r'<[^>]+>', ' ', str(text))
    return re.sub(r'http\S+', '', text)

# Aplicar limpieza a las nuevas columnas del esquema
df_mslearn['titulo'] = df_mslearn['titulo'].apply(clean_html_urls)
df_mslearn['texto'] = df_mslearn['texto'].apply(clean_html_urls)
df_mslearn['categoria'] = df_mslearn['categoria'].apply(clean_html_urls)
df_mslearn['subcategoria'] = df_mslearn['subcategoria'].apply(clean_html_urls)

# Eliminar duplicados basados en el contenido del 'texto'
initial_count = len(df_mslearn)
df_mslearn = df_mslearn.drop_duplicates(subset='texto').reset_index(drop=True)
print(f"Duplicados eliminados: {initial_count - len(df_mslearn)} (Filas restantes: {len(df_mslearn)})")

# Filtro de calidad: mínimo 30 palabras en la columna 'texto'
initial_count = len(df_mslearn)
df_mslearn = df_mslearn[df_mslearn['texto'].str.split().str.len() >= 30].reset_index(drop=True)
print(f"Textos con menos de 30 palabras eliminados: {initial_count - len(df_mslearn)} (Filas restantes: {len(df_mslearn)})")

print("\u2705 Limpieza de HTML, URLs, duplicados y textos cortos completada.")

Duplicados eliminados: 14 (Filas restantes: 4584)
Textos con menos de 30 palabras eliminados: 124 (Filas restantes: 4460)
✅ Limpieza de HTML, URLs, duplicados y textos cortos completada.


##  1.3 EDA

Cuantas filas hay por categoría

In [51]:
df_mslearn['categoria'].value_counts()

categoria
Aplicaciones empresariales    1405
nan                            979
Infraestructura                576
Seguridad                      440
Administración de datos        419
Inteligencia artificial        358
Desarrollo de aplicaciones     283
Name: count, dtype: int64

La categoría no da información suficiente para la categorización que se busca.

---
Cual es la Frecuencia por subcategoría

In [52]:
df_mslearn['subcategoria'].value_counts()

subcategoria
nan                                          2159
Computación en la nube                        212
Finanzas y contabilidad                       161
DevOps                                        132
Desarrollo de aplicaciones personalizadas     132
                                             ... 
solution-design                                 1
Riesgo interno                                  1
e-commerce                                      1
asset-management                                1
warehouse-management                            1
Name: count, Length: 62, dtype: int64

Tenemos 61 subcategorías y el caso donde no se registra subcategooría, esto nos da más información.

Veamos el top de tecnologías mencionadas individualmente.

In [53]:
individual_techs = df_mslearn['tecnologias_mencionadas'].str.split(',').explode().str.strip()
print("--- Top 20 tecnologías individuales más mencionadas ---")
print(individual_techs.value_counts().head(20))

--- Top 20 tecnologías individuales más mencionadas ---
tecnologias_mencionadas
microsoft                   1798
azure                       1437
datos                       1073
dynamics 365                 942
dynamics                     824
microsoft 365                559
copilot                      504
windows                      424
ia                           352
business central             331
microsoft power platform     327
desarrollo                   291
supply chain management      235
finance                      226
teams                        223
github                       221
office 365                   218
ms-copilot                   203
web                          198
microsoft teams              195
Name: count, dtype: int64


Al ser un dataset extraido de Microsoft Learn es normal que aparezcan los productos de esta empresa.

##  1.4.Mapeo de categorías (Lógica basada en Título, Texto, Tecnologías y Subcategoría)

In [54]:
# Diccionario 1: Palabras clave para búsqueda (Prioridad 1 y 2)
keywords_mapping = {
    'Backend': ['java', 'spring', 'c#', 'php', 'node.js', 'django', 'perl', 'ruby', 'scala', 'clojure', 'rust', 'haskell', 'elixir', 'earlang', 'flask', 'typescript', 'node', 'nodejs', 'laravel', 'api', 'rest', 'backend', 'csharp', 'dotnet', 'blockchain', 'cplusplus', 'c', 'go', 'express', 'graphql', 'kafka', 'solr', 'rabbitMQ', 'nginx', 'openresty', 'nestjs', 'firebase', '.net', 'rails'],
    'Bases de Datos': ['sql', 'mysql', 'mongodb', 'postgresql', 'redis', 'oracledb', 'cassandra', 'couchdb', 'hive', 'realm', 'mariadb', 'cockroachdb', 'elasticsearch', 'sqlite', 'mssql', 'sql server', 'sqlserver', 'cosmos db', 'database', 'bases de datos', 'nosql', 'storage'],
    'Cloud': ['aws', 'azure', 'oraclecloud', 'oci', 'googlecloud', 'gcp', 'nube', 'cloud', 'máquina virtual', 'virtual machine'],
    'Data Science': ['python', 'pandas', 'machine learning', 'deep', 'excel', 'llm', 'r', 'ia', 'deep learning', 'tensorflow', 'pytorch', 'numpy', 'seaborn', 'matplotlib', 'opencv', 'scikitlearn', 'scikit learn', 'd3js', 'chartjs', 'canvasjs', 'kibana', 'grafana', 'inteligencia artificial', 'datos', 'analytics', 'data analysis', 'aprendizaje automático', 'modelado de datos'],
    'DevOps': ['docker', 'kubernetes', 'ci/cd', 'devops', 'git', 'github', 'jenkins', 'bash', 'travisci', 'circleci', 'containers', 'automation'],
    'Frontend': ['javascript', 'html', 'html5', 'css', 'css3', 'react', 'angular', 'angularjs', 'vue', 'vuejs', 'scratch', 'frontend', 'svelte', 'backbonejs', 'bootstrap', 'vuetify', 'pug', 'gulp', 'sass', 'redux', 'webpack', 'babel', 'tailwind', 'materialize', 'bulma', 'gtk', 'qt', 'quasar', 'wxwidgets', 'wx widgets', 'ember', 'web'],
    'Mobile': ['objectivec', 'objective-c', 'android', 'ios', 'flutter', 'kotlin', 'swift', 'dart', 'nativescript', 'xamarin', 'reactnative', 'react native', 'ionic', 'apachecordova', 'mobile', 'multiplataforma']
}

# Diccionario 2: Mapeo directo por columna 'subcategoria' (Prioridad 3)
subcategory_to_category = {
    'Architecture': 'DevOps',
    'DevOps': 'DevOps',
    'Computación en la nube': 'Cloud',
    'Seguridad en la nube': 'Cloud',
    'Bases de datos': 'Bases de Datos',
    'Análisis de datos': 'Data Science',
    'Modelado de datos': 'Data Science',
    'Visualización de datos': 'Data Science',
    'Aprendizaje automático': 'Data Science',
    'Integración de datos': 'Data Science',
    'Ingeniería de datos': 'Data Science',
    'Protección contra amenazas': 'DevOps',
    'containers': 'DevOps'
}

def assign_final_category(row):
    # 1. Búsqueda en Título y Texto (Prioridad 1)
    combined_text = (str(row['titulo']) + " " + str(row['texto'])).lower()
    for category, keywords in keywords_mapping.items():
        for word in keywords:
            if re.search(r'\b' + re.escape(word) + r'\b', combined_text):
                return category

    # 2. Búsqueda en Tecnologías Mencionadas (Prioridad 2)
    techs_list = [t.strip().lower() for t in str(row['tecnologias_mencionadas']).split(',') if t.strip()]
    for category, keywords in keywords_mapping.items():
        if any(kw in techs_list for kw in keywords):
            return category

    # 3. Mapeo por Subcategoría (Prioridad 3)
    subcat = str(row['subcategoria'])
    if subcat in subcategory_to_category:
        return subcategory_to_category[subcat]

    return None

# Aplicar el nuevo mapeo sin usar la columna 'categoria' original
df_mslearn['categoria'] = df_mslearn.apply(assign_final_category, axis=1)

# Diagnóstico: tasa de coincidencia
total_rows = len(df_mslearn)
matched_rows = df_mslearn['categoria'].notna().sum()
print(f" Categorías mapeadas con éxito: {matched_rows} de {total_rows} ({matched_rows/total_rows*100:.1f}%)" if total_rows > 0 else "No hay datos para procesar.")

if total_rows > 0 and (matched_rows / total_rows) < 0.3:
    print("⚠️ ALERTA: La tasa de coincidencia es menor al 30%. Revisa el diccionario de mapeo.")

df_mslearn = df_mslearn.dropna(subset=['categoria']).reset_index(drop=True)

print("\n✅ Categorías asignadas.")
print("--- Distribución resultante tras el mapeo ---")
print(df_mslearn['categoria'].value_counts())


 Categorías mapeadas con éxito: 2913 de 4460 (65.3%)

✅ Categorías asignadas.
--- Distribución resultante tras el mapeo ---
categoria
Cloud             1174
Data Science       961
Bases de Datos     289
Backend            252
DevOps             131
Frontend            84
Mobile              22
Name: count, dtype: int64


Eliminar columnas auxiliares para finalizar el esquema

In [55]:
df_mslearn = df_mslearn.drop(columns=['subcategoria', 'tecnologias_mencionadas'])

print("✅ Columnas eliminadas.")
print(f"Columnas restantes: {df_mslearn.columns.tolist()}")
display(df_mslearn.head(3))

✅ Columnas eliminadas.
Columnas restantes: ['titulo', 'texto', 'categoria', 'autor', 'tipo']


,titulo,texto,categoria,autor,tipo
0,Experimento con Azure Machine Learning,Obtenga información sobre cómo encontrar el me...,Cloud,Microsoft,módulo
1,Implementación de la protección de datos del d...,En este módulo se describe cómo puede usar Int...,Data Science,Microsoft,módulo
2,Adición de lógica de decisión al código median...,Aprenda a bifurcar la ruta de acceso de ejecuc...,Backend,Microsoft,módulo


##  1.5. Balanceo de datos (Máximo 50 registros por categoría)

In [56]:
df_mslearn = pd.concat([
    group.sample(min(len(group), 50), random_state=42)
    for _, group in df_mslearn.groupby('categoria')
]).reset_index(drop=True)

print("✅ Datos balanceados (Máximo 50 registros por categoría).")

✅ Datos balanceados (Máximo 50 registros por categoría).


##  1.6. Exportación final y auditoría de calidad

In [57]:
final_df_mslearn = df_mslearn.copy()

# Selecciona solo las columnas que forman parte del esquema final de la base de datos
final_df_mslearn = final_df_mslearn[['titulo', 'texto', 'categoria', 'autor', 'tipo']]

print("=== DISTRIBUCIÓN FINAL POR CATEGORÍA ===")
print("   === DATASET - MICROSOFT LEARN ===")
category_counts = final_df_mslearn['categoria'].value_counts()
print(category_counts)

# Alerta si alguna categoría no alcanza el mínimo de 30 registros
print("\n--- Validación de Umbral Mínimo ---")
for cat, count in category_counts.items():
    if count < 30:
        print(f"⚠️ ADVERTENCIA: '{cat}' tiene {count} registros (Por debajo del mínimo de 30). Se requiere extracción manual.")

# Verifica que no haya columnas duplicadas antes de exportar
if final_df_mslearn.columns.duplicated().any():
    print("\n🚨 ERROR: hay columnas duplicadas en final_df_mslearn:")
    print(final_df_mslearn.columns[final_df_mslearn.columns.duplicated()].tolist())
else:
    print("\n✅ Sin columnas duplicadas — esquema limpio.")

final_df_mslearn.to_csv(f'{PROCESSED_FOLDER}/dataset_FINAL_mslearn.csv', index=False)
print("\n Pipeline completado. Dataset exportado con éxito al formato oficial.")

# Muestra 20 filas aleatorias para revisión manual de calidad
print("\n=== MUESTRA DE AUDITORÍA MANUAL (Revisión de 20 filas aleatorias) ===")
audit_sample = final_df_mslearn.sample(min(20, len(final_df_mslearn)), random_state=42)[['titulo', 'texto','categoria', 'autor', 'tipo']]
display(audit_sample)


=== DISTRIBUCIÓN FINAL POR CATEGORÍA ===
   === DATASET - MICROSOFT LEARN ===
categoria
Backend           50
Bases de Datos    50
Cloud             50
Data Science      50
DevOps            50
Frontend          50
Mobile            22
Name: count, dtype: int64

--- Validación de Umbral Mínimo ---
⚠️ ADVERTENCIA: 'Mobile' tiene 22 registros (Por debajo del mínimo de 30). Se requiere extracción manual.

✅ Sin columnas duplicadas — esquema limpio.

 Pipeline completado. Dataset exportado con éxito al formato oficial.

=== MUESTRA DE AUDITORÍA MANUAL (Revisión de 20 filas aleatorias) ===


,titulo,texto,categoria,autor,tipo
173,Creación de aplicaciones inteligentes y portal...,Cree aplicaciones y portales listos para intel...,Data Science,Microsoft,ruta de aprendizaje
132,"Visualizar, importar y exportar datos de Micro...",Esta ruta de aprendizaje le mostrará cómo usar...,Cloud,Microsoft,ruta de aprendizaje
197,Investigación de los riesgos de seguridad de d...,Investigue los riesgos de seguridad de datos m...,Data Science,Microsoft,módulo
9,Introducción a MongoDB API en Azure Cosmos DB,Obtenga información sobre los conceptos básico...,Backend,Microsoft,módulo
104,Supervisar y reparar de forma remota el equipa...,Configurar dispositivos de IoT para supervisar...,Cloud,Microsoft,módulo
119,Manage volume access for Azure NetApp Files,Learn how to manage access to a volume in Azur...,Cloud,Microsoft,módulo
256,Introducción al desarrollo web con Blazor,Evalúe si Blazor es adecuado para el siguiente...,Frontend,Microsoft,módulo
158,Acceso seguro a los datos en Microsoft Fabric,Obtenga información sobre los conceptos y estr...,Data Science,Microsoft,módulo
226,Entrega con DevOps,Compile y ejecute flujos de trabajo de integra...,DevOps,Microsoft,módulo
311,Implementar aplicaciones con Microsoft Intune ...,Este módulo le presenta la Microsoft Store par...,Mobile,Microsoft,módulo


# 3.Mezclar ambos datasets para obtener 50 por cada categoría

##  3.1.Carga de los sets de datos

In [58]:
file_path_coursera = f'{PROCESSED_FOLDER}/dataset_FINAL_coursera.csv'
file_path_mslearn = f'{PROCESSED_FOLDER}/dataset_FINAL_mslearn.csv'

# Carga segura con alternativa de codificación (encoding fallback) en caso de error
try:
    df_coursera = pd.read_csv(file_path_coursera, encoding='utf-8')
    df_mslearn = pd.read_csv(file_path_mslearn, encoding='utf-8')
except UnicodeDecodeError:
    df_coursera = pd.read_csv(file_path_coursera, encoding='latin1')
    df_mslearn = pd.read_csv(file_path_mslearn, encoding='latin1')

Unificar datasets, balanceo de 50 registros

In [59]:
# 1. Unificar ambos datasets
df_unified = pd.concat([df_coursera, df_mslearn], ignore_index=True)

# 2. Balanceo final: Obtener 50 registros por categoría (o el máximo posible)
df_final_unified = pd.concat([
    group.sample(min(len(group), 50), random_state=42)
    for _, group in df_unified.groupby('categoria')
]).reset_index(drop=True)

##  3.3.Exportación final y auditoría de calidad

In [60]:
final_columns = ['titulo', 'texto', 'categoria', 'autor','tipo']
df_final_unified = df_final_unified[final_columns]

print("=== DISTRIBUCIÓN FINAL DEL DATASET UNIFICADO ===")
category_counts = df_final_unified['categoria'].value_counts()
print(category_counts)

# Validación de Umbral
print("\n--- Validación de Umbral Mínimo (30 registros) ---")
for cat, count in category_counts.items():
    if count < 30:
        print(f"⚠️ ADVERTENCIA: '{cat}' tiene {count} registros. Se recomienda revisión manual.")
    else:
        print(f"✅ '{cat}': OK ({count} registros)")

# Exportación
output_file = f'{PROCESSED_FOLDER}/dataset_FINAL_UNIFICADO_techmind.csv'
df_final_unified.to_csv(output_file, index=False)

print(f"\n✅ Pipeline de unificación completado. Archivo guardado en: {output_file}")

# Muestra de Auditoría
print("\n=== MUESTRA DE AUDITORÍA (20 registros aleatorios) ===")
display(df_final_unified.sample(min(20, len(df_final_unified)), random_state=42))

=== DISTRIBUCIÓN FINAL DEL DATASET UNIFICADO ===
categoria
Backend           50
Bases de Datos    50
Cloud             50
Data Science      50
DevOps            50
Frontend          50
Mobile            50
Name: count, dtype: int64

--- Validación de Umbral Mínimo (30 registros) ---
✅ 'Backend': OK (50 registros)
✅ 'Bases de Datos': OK (50 registros)
✅ 'Cloud': OK (50 registros)
✅ 'Data Science': OK (50 registros)
✅ 'DevOps': OK (50 registros)
✅ 'Frontend': OK (50 registros)
✅ 'Mobile': OK (50 registros)

✅ Pipeline de unificación completado. Archivo guardado en: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\procesados/dataset_FINAL_UNIFICADO_techmind.csv

=== MUESTRA DE AUDITORÍA (20 registros aleatorios) ===


,titulo,texto,categoria,autor,tipo
157,Configurar los agentes de IA y Copilot en Dyna...,Transforma tu centro de contacto con IA. Confi...,Data Science,Microsoft,módulo
341,Inscripción de dispositivos mediante Microsoft...,Los alumnos aprenderán a configurar y configur...,Mobile,Microsoft,módulo
315,Pruebas de accesibilidad web con Accessibility...,En este curso basado en proyectos de 2 horas d...,Mobile,Coursera Project Network,articulo
234,Automatización de las tareas de desarrollo med...,Cree una acción de GitHub básica y úsela en un...,DevOps,Microsoft,módulo
155,"Python: imputaciones, creación de funciones y ...",En este curso basado en proyectos de 3 horas y...,Data Science,Coursera Project Network,articulo
274,Extensión de Microsoft Viva Connections con pe...,Obtenga información sobre cómo ampliar Viva Co...,Frontend,Microsoft,módulo
304,Principios de marketing y medios digitales,La revolución digital ha provocado un cambio t...,Mobile,University of Illinois at Urbana-Champaign,articulo
227,Identificación de la deuda técnica,Obtenga información sobre cómo buscar y admini...,DevOps,Microsoft,módulo
278,Imágenes y enlaces en HTML,"En este proyecto, escribirá el código HTML par...",Frontend,Coursera Project Network,articulo
185,Conceptos básicos de visión por computadora,"Al final de este curso, los estudiantes compre...",Data Science,The State University of New York,articulo
